# 07 — Does constraint learning transfer to Go?

**Purpose.** Prepare a frozen-model test of CSP/QCSP-only pretraining against the same AHR architecture at its original random initialization, random legal play, and a simple area-greedy control. This notebook performs no optimization and does not change weights.

**Default Run All:** validate the evaluator on development fixtures, run tiny untrained 3×3/9×9 wiring smokes, verify interruption/resume and write an honest `NOT_RUN_NO_PRETRAINED_CHECKPOINT` receipt. Smoke results do not measure learned transfer. Expected runtime: under 3 minutes on one CPU; no GPU required.

**Inputs:** tagged definitions from notebooks 00–06; optionally a hash-verified, CSP-only checkpoint from notebook 05. **Outputs:** `reports/07_transfer_validation.json`, an immutable development protocol, and resumable evaluation checkpoints under `checkpoints/transfer-07-<identity>/`. No final/lockbox file is opened.

**Run a real pilot later:** only after CSP pretraining is separately authorized and completed, select its run directory with `AHR07_CSP_RUN`, then set `AHR07_RUN_TRANSFER=1` and Restart Kernel and Run All. This is a development pilot; preregister a separately protected final evaluation before making a final claim. Defaults keep this route off.

**Exact resume:** keep the same notebook definitions, model checkpoint and protocol, then Restart Kernel and Run All with the same environment variables. The manifest records the last completed position/game ply and resume cell. The two newest valid snapshots and the reference snapshot are retained. Invalid manifests, hashes, changed models or changed protocols stop execution. All frozen parameters, RNG streams, board/superko history, move traces and job cursors are restored; no optimizer or in-flight search exists at the committed boundary. Timings may differ after restart; the deterministic smoke checks exact decisions and metrics apart from timing.

The workspace is on the persistent boot disk; instance deletion policy remains unverified. Copy final/best artifacts off-instance and verify hashes before deletion.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, time, math, copy, random
import nbformat
import numpy as np
import torch

PROJECT_ROOT = Path(os.environ.get("AHR_GO_PROJECT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()
if not (PROJECT_ROOT / "notebooks/00_instance_setup_and_recovery.ipynb").is_file():
    raise RuntimeError("Run from ahr_go or its notebooks directory, or set AHR_GO_PROJECT.")
for filename in ["00_instance_setup_and_recovery.ipynb", "01_go_rules_and_independent_verifier.ipynb",
                 "02_go_to_csp_and_qcsp.ipynb", "03_curricula_datasets_and_splits.ipynb",
                 "04_native_ahr_architecture.ipynb", "05_csp_qcsp_pretraining.ipynb", "06_ahr_go_search_engine.ipynb"]:
    notebook = nbformat.read(PROJECT_ROOT / "notebooks" / filename, as_version=4)
    for index, cell in enumerate(notebook.cells):
        if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", []):
            exec(compile(cell.source, f"{filename}:export:{index}", "exec"), globals())
torch.set_num_threads(1)
TRANSFER_RUN_REQUESTED = os.environ.get("AHR07_RUN_TRANSFER", "0") == "1"
SELECTED_CSP_RUN = os.environ.get("AHR07_CSP_RUN", "").strip()
print({"project": str(PROJECT_ROOT), "training": False, "transfer_pilot_requested": TRANSFER_RUN_REQUESTED,
       "data_scope": "new development fixtures only; final/lockbox untouched"})

## Immutable protocol and matched controls

Commit the protocol before any evaluated move. Each candidate faces the same opponent, starting state, budget and seed twice with swapped colors. The untrained AHR control uses the exact pretraining initialization, not a newly chosen favorable seed. Random and greedy controls are development diagnostics, not alternate AHR methods. All competitors use the same legal rules, whole-call budget and move cap.

Position probes ask a **bounded horizon-two area question**. A returned move fails that probe only when another action is verified SAT and its own continuation is verified UNSAT. Cutoffs remain UNKNOWN. This label is not a full-game good/bad-move oracle.

In [ ]:
TRANSFER_SCHEMA = "ahr07-frozen-transfer-v1"
TRANSFER_RESUME_CELL = "07: Execute or resume frozen development evaluation — Restart Kernel and Run All"

def transfer_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)

def transfer_digest(value):
    return hashlib.sha256(transfer_json(value).encode()).hexdigest()

def transfer_source_identity(project):
    result = {}
    for number in range(8):
        paths = list((Path(project) / "notebooks").glob(f"{number:02d}_*.ipynb"))
        if len(paths) != 1:
            raise RuntimeError(f"Expected exactly one notebook {number:02d}")
        notebook = nbformat.read(paths[0], as_version=4)
        result[paths[0].name] = hashlib.sha256("\n\n".join(cell.source for cell in notebook.cells if cell.cell_type == "code").encode()).hexdigest()
    return result

def transfer_protocol(*, sources, models, candidates, seeds=(113, 227), budgets=(0.05,), size=3, komi=0.5,
                      max_moves=40, opponents=("random_legal", "greedy_area"), positions=True, label="development_pilot"):
    if label not in ("development_pilot", "wiring_smoke", "recovery_fixture"):
        raise ValueError("This notebook cannot open or run final/lockbox evaluation")
    if not candidates or len(set(candidates)) != len(candidates) or not seeds or len(set(seeds)) != len(seeds):
        raise ValueError("Require distinct candidates and seeds")
    if not budgets or len(set(budgets)) != len(budgets) or not opponents or len(set(opponents)) != len(opponents):
        raise ValueError("Require distinct nonempty budgets and opponents")
    if any(type(seed) is not int for seed in seeds) or any(isinstance(x, bool) or not math.isfinite(float(x)) or x <= 0 for x in budgets):
        raise ValueError("Seeds and budgets are invalid")
    if type(max_moves) is not int or max_moves < 2 or max(budgets) > 10:
        raise ValueError("Use a positive bounded move cap and at most ten seconds per call")
    initial_state(size, komi)
    return {"schema": TRANSFER_SCHEMA, "label": label, "data_split": "development", "sources": sources,
            "models": models, "candidates": list(candidates), "seeds": list(seeds), "budgets_s": list(budgets),
            "board_size": size, "komi": komi, "max_moves": max_moves, "opponents": list(opponents),
            "position_probes": bool(positions), "colors": [1, -1], "rules": copy.deepcopy(RULESET),
            "pairing": "same seed, opponent, opening and budget; swap candidate colors",
            "fixture_origin": "new seeded legal development replays; no dataset shards read",
            "position_horizon": 2, "position_max_nodes": 10000,
            "timeout_policy": "UNKNOWN; do not award a win", "cutoff_policy": "UNKNOWN; do not score unfinished boards",
            "primary_metric": "paired candidate minus original-initialization AHR game score, complete matched seed blocks",
            "uncertainty": "95% paired seed-block bootstrap; descriptive development estimate",
            "minimum_blocks_for_interval": 3, "bootstrap_resamples": 2000,
            "decision_policy": "development only; no full-game optimality, Elo or final transfer claim"}

def transfer_fixture(protocol, seed, position=False):
    rng = random.Random(int(transfer_digest(["ahr07-dev-v1", seed, position])[:16], 16))
    state = initial_state(protocol["board_size"], protocol["komi"])
    for _ in range(2 if position else 0):
        choices = [action for action in legal_actions(state) if action != state.pass_action]
        if not choices:
            break
        state = play(state, rng.choice(choices))
    return state

def transfer_jobs(protocol):
    jobs = []
    for seed in protocol["seeds"]:
        for budget in protocol["budgets_s"]:
            for candidate in protocol["candidates"]:
                if protocol["position_probes"]:
                    jobs.append({"kind": "position", "candidate": candidate, "seed": seed, "budget_s": budget})
                for opponent in protocol["opponents"]:
                    for color in protocol["colors"]:
                        jobs.append({"kind": "game", "candidate": candidate, "opponent": opponent, "seed": seed,
                                     "budget_s": budget, "candidate_color": color})
    return [dict(job, job_id=transfer_digest(job)[:20]) for job in jobs]

def transfer_commit_protocol(run_dir, protocol):
    path = Path(run_dir) / "protocol.json"
    if path.exists():
        if json.loads(path.read_text()) != protocol:
            raise RuntimeError("Protocol changed after commitment; use a distinct run ID")
    elif (Path(run_dir) / "manifest.json").exists() or list(Path(run_dir).glob("*.pt")):
        raise RuntimeError("Evaluation checkpoints exist without their protocol")
    else:
        atomic_json(path, protocol)
    return transfer_digest(protocol)


## Frozen players and independent evidence

The model is placed in `eval()` mode; autograd and all optimizer updates are disabled. Parameter/buffer hashes are checked before and after evaluation. A checkpoint trained on Go, a missing provenance record or a changed pretraining initialization is rejected. Every emitted action is checked independently before it is applied. A deadline miss, illegal proposal, move cap and a completed two-pass game receive different statuses.

Latency is measured around the player call, including notebook06’s final legality audit. The evaluator’s additional independent cross-check runs outside that timer for every player.

In [ ]:
def transfer_baseline(name):
    if name not in ("random_legal", "greedy_area", "pass_fixture"):
        raise ValueError("Unknown baseline")
    def decide(state, time_budget_s, seed):
        choices = legal_actions(state)
        if not choices:
            return {"action": None, "status": "TERMINAL"}
        if name == "pass_fixture":
            action = state.pass_action
        elif name == "random_legal":
            action = random.Random(seed).choice(choices)
        else:
            scored = [(state.to_play * area_score(play(state, action)), action) for action in choices]
            best = max(score for score, _ in scored)
            action = random.Random(seed).choice([action for score, action in scored if score == best])
        return {"action": action, "status": "BASELINE", "model": name}
    return decide

def transfer_reference_board(state, action):
    """Independent coordinate implementation: no play/group helper is reused."""
    if type(action) is not int or state.terminal or not 0 <= action <= state.pass_action:
        return None
    if action == state.pass_action:
        return state.board
    n = state.size
    grid = {(r,c): state.board[r*n+c] for r in range(n) for c in range(n)}
    point = divmod(action, n)
    if grid[point]:
        return None
    grid[point] = state.to_play
    def adjacent(point):
        r,c = point
        return [p for p in ((r-1,c),(r+1,c),(r,c-1),(r,c+1)) if p in grid]
    def component(point):
        found, todo, liberties = {point}, [point], set()
        for p in todo:
            for q in adjacent(p):
                if grid[q] == 0:
                    liberties.add(q)
                elif grid[q] == grid[point] and q not in found:
                    found.add(q); todo.append(q)
        return found, liberties
    captured = set()
    for neighbor in adjacent(point):
        if grid[neighbor] == -state.to_play:
            stones, liberties = component(neighbor)
            if not liberties:
                captured.update(stones)
    for p in captured:
        grid[p] = 0
    if not component(point)[1]:
        return None
    board = tuple(grid[(r,c)] for r in range(n) for c in range(n))
    return None if board in state.history else board

def transfer_reference_score(state):
    total = -float(state.komi)
    for start, occupant in enumerate(state.board):
        if occupant:
            total += occupant
            continue
        visited, pending, borders = {start}, [start], set()
        for point in pending:
            r,c = divmod(point, state.size)
            for rr,cc in ((r-1,c),(r+1,c),(r,c-1),(r,c+1)):
                if 0 <= rr < state.size and 0 <= cc < state.size:
                    other = rr*state.size+cc
                    color = state.board[other]
                    if color:
                        borders.add(color)
                    elif other not in visited:
                        visited.add(other); pending.append(other)
        if len(borders) == 1:
            total += next(iter(borders))
    return total

def transfer_call(player, state, budget, seed):
    start = time.perf_counter()
    with torch.inference_mode():
        decision = player(state, budget, seed)
    elapsed = time.perf_counter() - start
    if not isinstance(decision, dict):
        raise TypeError("Player adapter must return a dictionary")
    action = decision.get("action")
    independent = transfer_reference_board(state, action)
    legal = independent is not None
    if legal and play(state, action).board != independent:
        raise RuntimeError("Independent legality checker disagrees with production rules")
    return {"action": action, "player_status": str(decision.get("status", "UNSPECIFIED")),
            "elapsed_s": elapsed, "time_budget_s": budget, "deadline_missed": elapsed > budget,
            "legal": legal, "metadata": {key: decision[key] for key in ("nodes", "micro_steps", "reason", "strategic_status", "fallback_pass", "rng_probe") if key in decision}}

def transfer_position_result(protocol, job, player):
    state = transfer_fixture(protocol, job["seed"], True)
    move = transfer_call(player, state, job["budget_s"], job["seed"])
    result = dict(job, move=move, state=state.to_dict(), status="UNKNOWN", bounded_mistake=None)
    if not move["legal"]:
        result["status"] = "ERROR_ILLEGAL_PROPOSAL"
        return result
    if move["deadline_missed"]:
        result["status"] = "UNKNOWN_TIMEOUT"
        return result
    objective = GoObjective("area_at_horizon", state.to_play, state.to_play * area_score(state))
    evidence = {}
    for action in legal_actions(state):
        outcome = solve_go_bounded(play(state, action), protocol["position_horizon"]-1, objective,
                                   max_nodes=protocol["position_max_nodes"])
        evidence[str(action)] = outcome.status
    chosen = evidence[str(move["action"])]
    exists_verified = SAT in evidence.values()
    result.update(status="VERIFIED_BOUNDED_PROBE", objective=asdict(objective), horizon=protocol["position_horizon"],
                  action_verdicts=evidence, chosen_verdict=chosen,
                  bounded_mistake=(chosen == UNSAT) if exists_verified and chosen != UNKNOWN else None,
                  scope="horizon-two area objective only; no full-game optimality claim")
    return result


## Interruption-safe evaluation

Each position and each completed move is a stage boundary and is saved immediately. One writer owns a run for its whole session. Frozen model weights and initialization are included so a snapshot contains everything needed to replay from the next move. The `best` pointer denotes the reference snapshot; no evaluation-selected model is promoted. Sessions return at a bounded time limit and resume from their manifest.

In [ ]:
def transfer_validate_snapshot(state, protocol, model_digests):
    required = REQUIRED_CHECKPOINT_KEYS | {"schema", "protocol_digest", "optimizer_steps", "models_digest", "model_configs", "results", "active", "complete"}
    if not isinstance(state, dict) or required - set(state):
        raise RuntimeError("Incomplete transfer checkpoint")
    if state["schema"] != TRANSFER_SCHEMA or state["protocol_digest"] != transfer_digest(protocol):
        raise RuntimeError("Transfer checkpoint protocol/schema mismatch")
    if set(state["model_configs"]) != set(model_digests) or set(state["model"]) != set(model_digests):
        raise RuntimeError("Checkpoint model/config inventory mismatch")
    for config in state["model_configs"].values():
        AHRConfig(**config)
    if state["models_digest"] != model_digests or state["config"] != protocol or state["optimizer_steps"] != 0:
        raise RuntimeError("Frozen model identity/configuration changed")
    if any(state[key] is not None for key in ("optimizer", "scheduler", "scaler")):
        raise RuntimeError("Transfer evaluation cannot contain optimizer state")
    cursor = state["sampler"].get("next_job")
    if type(cursor) is not int or not 0 <= cursor <= len(transfer_jobs(protocol)) or len(state["results"]) != cursor:
        raise RuntimeError("Evaluation cursor/results mismatch")
    if state["complete"] != (cursor == len(transfer_jobs(protocol))) or type(state["global_step"]) is not int:
        raise RuntimeError("Evaluation completion/step mismatch")
    if state["active"] is not None:
        active = state["active"]
        board = GoState.from_dict(active["state"])
        if active["job_id"] != transfer_jobs(protocol)[cursor]["job_id"] or board.move_number != len(active["moves"]):
            raise RuntimeError("In-flight game cursor is inconsistent")

def transfer_load_unlocked(run_dir, protocol, model_digests):
    directory = Path(run_dir)
    manifest_path = directory / "manifest.json"
    if not manifest_path.exists():
        if list(directory.glob("*.pt")):
            raise RuntimeError("Transfer payload exists without a manifest; refusing restart")
        return None
    try:
        manifest = json.loads(manifest_path.read_text())
        entries = manifest["entries"]
        if not entries or manifest["schema_version"] != 1 or manifest["resume_cell"] != TRANSFER_RESUME_CELL:
            raise ValueError("Invalid manifest schema")
        if len({entry["file"] for entry in entries}) != len(entries) or len({entry["global_step"] for entry in entries}) != len(entries):
            raise ValueError("Duplicate checkpoint entries")
        latest = max(entries, key=lambda entry: entry["global_step"])
        if manifest["latest"] != latest["file"] or manifest["best"] not in {entry["file"] for entry in entries}:
            raise ValueError("Manifest pointer mismatch")
        recovered = None
        for entry in entries:
            payload = _checkpoint_entry_state(directory, entry)
            if payload is None:
                raise ValueError("A committed checkpoint failed its hash/schema validation")
            transfer_validate_snapshot(payload, protocol, model_digests)
            if entry["file"] == latest["file"]:
                recovered = payload
        return recovered
    except Exception as error:
        raise RuntimeError("Invalid transfer checkpoint; refusing silent restart or fallback") from error

def transfer_snapshot(models, protocol):
    return {"schema": TRANSFER_SCHEMA, "model": {name: architecture_cpu_tree(model.state_dict()) for name, model in models.items()},
            "model_configs": {name: asdict(model.config) for name, model in models.items()},
            "adapters": None, "optimizer": None, "scheduler": None, "scaler": None, "optimizer_steps": 0,
            "rng": capture_rng(), "sampler": {"next_job": 0}, "global_step": 0,
            "global_step_semantics": "evaluation stage boundaries; never optimizer steps", "config": protocol,
            "metrics": {"positions": 0, "game_plies": 0, "games": 0}, "protocol_digest": transfer_digest(protocol),
            "models_digest": {name: architecture_parameter_digest(model) for name, model in models.items()},
            "results": [], "active": None, "complete": False,
            "replay": None, "distributed": {"world_size": 1, "rank": 0},
            "best_semantics": "reference frozen initialization snapshot; no best-model selection"}

def transfer_run(project, run_id, protocol, players, models, *, stop_after_stages=None, session_seconds=600):
    if Path(run_id).name != run_id or run_id in (".", "..") or not run_id:
        raise ValueError("Use a simple nonempty run ID")
    if not 0 < session_seconds <= 840:
        raise ValueError("Bound each session below fifteen minutes")
    for model in models.values():
        model.eval()
        model.requires_grad_(False)
    expected = {name: architecture_parameter_digest(model) for name, model in models.items()}
    if protocol["models"] != expected:
        raise ValueError("Protocol model hashes do not match frozen inputs")
    jobs = transfer_jobs(protocol)
    if set(protocol["candidates"] + protocol["opponents"]) - set(players):
        raise ValueError("A protocol player is missing")
    directory = Path(project) / "checkpoints" / run_id
    started, advanced = time.monotonic(), 0
    with checkpoint_lock(directory):
        transfer_commit_protocol(directory, protocol)
        state = transfer_load_unlocked(directory, protocol, expected)
        resumed_step = None if state is None else state["global_step"]
        if state is None:
            state = transfer_snapshot(models, protocol)
            _save_checkpoint_unlocked(directory, state, best=True, resume_cell=TRANSFER_RESUME_CELL)
        else:
            for name, model in models.items():
                if asdict(model.config) != state["model_configs"][name]:
                    raise RuntimeError("Restored frozen architecture configuration changed")
                model.load_state_dict(state["model"][name], strict=True)
                if architecture_parameter_digest(model) != expected[name]:
                    raise RuntimeError("Restored frozen parameters do not match protocol")
            if len(state["rng"]["torch_cuda"]) != torch.cuda.device_count():
                raise RuntimeError("Exact evaluation resume needs the saved CUDA topology")
            restore_rng(state["rng"])
        while not state["complete"]:
            if (stop_after_stages is not None and advanced >= stop_after_stages) or time.monotonic()-started >= session_seconds:
                break
            job = jobs[state["sampler"]["next_job"]]
            outcome = None
            if job["kind"] == "position":
                outcome = transfer_position_result(protocol, job, players[job["candidate"]])
                state["metrics"]["positions"] += 1
            else:
                if state["active"] is None:
                    position = transfer_fixture(protocol, job["seed"])
                    state["active"] = {"job_id": job["job_id"], "state": position.to_dict(), "moves": []}
                active = state["active"]
                position = GoState.from_dict(active["state"])
                player_name = job["candidate"] if position.to_play == job["candidate_color"] else job["opponent"]
                # Derive a fresh stream per role/ply: order and restart cannot change randomness.
                player_seed = int(transfer_digest([job["seed"], position.move_number, player_name])[:16], 16)
                move = transfer_call(players[player_name], position, job["budget_s"], player_seed)
                move.update(player=player_name, color=position.to_play, is_candidate_turn=position.to_play == job["candidate_color"])
                active["moves"].append(move)
                state["metrics"]["game_plies"] += 1
                status, score = None, None
                if not move["legal"]:
                    status = "ERROR_ILLEGAL_PROPOSAL"
                elif move["deadline_missed"]:
                    status = "UNKNOWN_TIMEOUT"
                else:
                    position = play(position, move["action"])
                    active["state"] = position.to_dict()
                    if position.terminal:
                        status = "COMPLETED_TWO_PASSES"
                        score = area_score(position)
                        if score != transfer_reference_score(position):
                            raise RuntimeError("Independent final area score disagrees")
                    elif len(active["moves"]) >= protocol["max_moves"]:
                        status = "UNKNOWN_MOVE_LIMIT"
                if status is not None:
                    candidate_score = None if score is None else (0.5 if score == 0 else float(score*job["candidate_color"] > 0))
                    outcome = dict(job, status=status, moves=active["moves"], final_state=position.to_dict(),
                                   black_minus_white=score, candidate_score=candidate_score,
                                   winner=None if score is None or score == 0 else (1 if score > 0 else -1))
                    state["metrics"]["games"] += 1
            if outcome is not None:
                state["results"].append(outcome)
                state["sampler"]["next_job"] += 1
                state["active"] = None
            if any(architecture_parameter_digest(model) != expected[name] for name, model in models.items()):
                raise RuntimeError("Frozen parameters changed during evaluation")
            state["complete"] = state["sampler"]["next_job"] == len(jobs)
            state["global_step"] += 1
            state["rng"] = capture_rng()
            transfer_validate_snapshot(state, protocol, expected)
            _save_checkpoint_unlocked(directory, state, resume_cell=TRANSFER_RESUME_CELL)
            advanced += 1
        return state, {"resumed_from_step": resumed_step, "new_stages": advanced, "complete": state["complete"],
                       "frozen_hashes_verified": expected, "run_id": run_id}


## Report uncertainty and incomplete evidence

Only naturally terminated two-pass games have winners. The all-scheduled-game score bounds assign unresolved games every possible score in [0, 1], exposing survivor bias. Confidence intervals resample whole seed blocks across colors and opponents; they do not treat correlated paired games as independent observations. Tiny smoke games never generate Elo or a transfer-improvement claim.

In [ ]:
def transfer_summarize(protocol, results):
    from collections import Counter
    summary = {"schema": TRANSFER_SCHEMA, "label": protocol["label"], "completed_jobs": len(results),
               "planned_jobs": len(transfer_jobs(protocol)), "status_counts": dict(Counter(row["status"] for row in results)),
               "candidates": {}, "paired_effects": {}, "training_performed": False, "optimizer_steps": 0,
               "final_claim": "NOT_ESTABLISHED", "elo": None}
    for candidate in protocol["candidates"]:
        games = [row for row in results if row["kind"] == "game" and row["candidate"] == candidate]
        complete = [row for row in games if row["candidate_score"] is not None]
        planned = sum(job["kind"] == "game" and job["candidate"] == candidate for job in transfer_jobs(protocol))
        score_sum = sum(row["candidate_score"] for row in complete)
        moves = [move for row in games for move in row["moves"]]
        probes = [row for row in results if row["kind"] == "position" and row["candidate"] == candidate]
        judged = [row for row in probes if row["bounded_mistake"] is not None]
        summary["candidates"][candidate] = {"completed_games": len(complete), "recorded_games": len(games),
            "planned_games": planned, "completed_game_mean_score": score_sum/len(complete) if complete else None,
            "all_planned_score_bounds": [score_sum/planned, (score_sum+planned-len(complete))/planned] if planned else None,
            "moves": len(moves), "illegal_proposals": sum(not move["legal"] for move in moves),
            "deadline_misses": sum(move["deadline_missed"] for move in moves),
            "mean_call_seconds": float(np.mean([move["elapsed_s"] for move in moves])) if moves else None,
            "p95_call_seconds": float(np.quantile([move["elapsed_s"] for move in moves], .95)) if moves else None,
            "candidate_moves": sum(move["is_candidate_turn"] for move in moves),
            "candidate_illegal_proposals": sum(not move["legal"] for move in moves if move["is_candidate_turn"]),
            "candidate_deadline_misses": sum(move["deadline_missed"] for move in moves if move["is_candidate_turn"]),
            "candidate_fallback_passes": sum(bool(move["metadata"].get("fallback_pass", False)) for move in moves if move["is_candidate_turn"]),
            "candidate_mean_call_seconds": float(np.mean([move["elapsed_s"] for move in moves if move["is_candidate_turn"]])) if any(move["is_candidate_turn"] for move in moves) else None,
            "bounded_judged_positions": len(judged), "bounded_mistakes": sum(row["bounded_mistake"] for row in judged),
            "bounded_unknown_chosen_positions": sum(row.get("chosen_verdict") == UNKNOWN for row in probes),
            "bounded_inapplicable_or_unjudged_positions": len(probes)-len(judged),
            "probe_scope": "horizon-two area objective; UNKNOWN excluded and counted in statuses"}
    if "csp_pretrained_ahr" in protocol["candidates"] and "untrained_ahr" in protocol["candidates"]:
        for budget in protocol["budgets_s"]:
            lookup = {(row["candidate"], row["seed"], row["opponent"], row["candidate_color"]): row["candidate_score"]
                      for row in results if row["kind"] == "game" and row["budget_s"] == budget}
            blocks = []
            for seed in protocol["seeds"]:
                pairs = [(lookup.get(("csp_pretrained_ahr", seed, opponent, color)), lookup.get(("untrained_ahr", seed, opponent, color)))
                         for opponent in protocol["opponents"] for color in protocol["colors"]]
                if all(a is not None and b is not None for a,b in pairs):
                    blocks.append(float(np.mean([a-b for a,b in pairs])))
            interval = None
            if len(blocks) >= protocol["minimum_blocks_for_interval"]:
                rng = np.random.default_rng(20260923)
                draws = rng.choice(blocks, size=(protocol["bootstrap_resamples"], len(blocks)), replace=True).mean(1)
                interval = np.quantile(draws, [.025,.975]).tolist()
            summary["paired_effects"][str(budget)] = {"complete_seed_blocks": len(blocks), "planned_seed_blocks": len(protocol["seeds"]),
                "candidate_minus_untrained_mean": float(np.mean(blocks)) if blocks else None,
                "paired_seed_bootstrap_95_interval": interval,
                "interpretation": "descriptive development evidence only; missing blocks may bias complete-case estimate"}
    return summary

def transfer_without_timing(value):
    if isinstance(value, dict):
        return {key: transfer_without_timing(item) for key,item in value.items() if key != "elapsed_s"}
    if isinstance(value, list):
        return [transfer_without_timing(item) for item in value]
    return value


## Validation and current model availability

The next cells test completed games, timeout/illegal/cutoff distinctions, frozen hashes, color pairing, immutable protocol, corrupted checkpoint rejection and exact continuation. A separate tiny untrained AHR run verifies the real engine wiring. Neither fixture supplies evidence for the CSP-to-Go learning hypothesis.

In [ ]:
def transfer_seed_recovery():
    random.seed(90217)
    np.random.seed(90217)
    torch.manual_seed(90217)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(90217)

def transfer_recovery_player(state, budget, seed):
    choices = [action for action in legal_actions(state) if action != state.pass_action]
    action = random.Random(seed).choice(choices) if state.move_number < 2 and choices else state.pass_action
    probe = [random.random(), float(np.random.random()), float(torch.rand(()))]
    return {"action": action, "status": "RECOVERY_FIXTURE", "rng_probe": probe}

def transfer_ahr_player(model, codec, search_config=None):
    model.eval()
    model.requires_grad_(False)
    def decide(state, time_budget_s, seed):
        # The 06 engine is deterministic for fixed model/state and work completed.
        result = choose_ahr_go_move(state, model, codec, time_budget_s=time_budget_s, search_config=search_config)
        return {"action": result["action"], "status": result["status"],
                "nodes": result.get("counters", {}).get("nodes"),
                "micro_steps": result.get("counters", {}).get("model_microsteps"),
                "reason": result.get("reason", "unspecified"),
                "strategic_status": result.get("strategic_status", "UNKNOWN"),
                "fallback_pass": result.get("fallback_pass", False)}
    return decide

def transfer_load_frozen_pair(project, run_id):
    model, codec, provenance = load_pretrained_ahr(project, run_id, device="cpu", selection="best")
    if provenance.get("training_route") != "csp_only" or provenance.get("trained_on_go") is not False:
        raise RuntimeError("Transfer hypothesis requires CSP-only provenance and no Go training")
    if provenance.get("purpose") != "csp_pretraining" or type(provenance.get("trained_optimizer_steps")) is not int or provenance["trained_optimizer_steps"] <= 0:
        raise RuntimeError("A synthetic smoke or untrained checkpoint cannot represent pretraining")
    initial = provenance.get("initial_model_state")
    if not isinstance(initial, dict) or not initial:
        raise RuntimeError("Original pretraining initialization is required for a matched control")
    untrained = NativeAHR(model.config).cpu()
    untrained.load_state_dict(initial, strict=True)
    for frozen in (model, untrained):
        frozen.eval()
        frozen.requires_grad_(False)
    if architecture_parameter_digest(model) == architecture_parameter_digest(untrained):
        raise RuntimeError("Pretrained and initialization parameters are identical")
    # Full metadata includes tensors; report only a digest identity and declared route.
    identity = {key: value for key,value in provenance.items() if key != "initial_model_state"}
    return model, untrained, codec, identity

def transfer_kernel_recovery(project, protocol, run_id, phase):
    """Fresh-kernel replay without authoring any executable helper file."""
    from nbclient import NotebookClient
    if phase not in ("interrupt", "resume"):
        raise ValueError("Unknown recovery phase")
    script = """from pathlib import Path
import nbformat, json
PROJECT_ROOT = Path(PROJECT_TEXT)
for number in (0, 1, 2, 4, 7):
    path = next((PROJECT_ROOT / 'notebooks').glob(f'{number:02d}_*.ipynb'))
    for cell in nbformat.read(path, as_version=4).cells:
        if cell.cell_type == 'code' and 'ahr-export' in cell.metadata.get('tags', []):
            exec(compile(cell.source, str(path), 'exec'), globals())
import torch
torch.set_num_threads(1)
protocol = json.loads(PROTOCOL_TEXT)
transfer_seed_recovery()
state, receipt = transfer_run(PROJECT_ROOT, RUN_TEXT, protocol,
    {'recovery_fixture': transfer_recovery_player}, {}, stop_after_stages=STOP_TEXT)
atomic_json(PROJECT_ROOT / 'reports' / REPORT_TEXT, {'receipt': receipt, 'results': transfer_without_timing(state['results']), 'metrics': state['metrics']})
"""
    replacements = {"PROJECT_TEXT": repr(str(project)), "PROTOCOL_TEXT": repr(transfer_json(protocol)),
                    "RUN_TEXT": repr(run_id), "STOP_TEXT": "1" if phase == "interrupt" else "None",
                    "REPORT_TEXT": repr("07_recovery_" + phase + ".json")}
    for old,new in replacements.items():
        script = script.replace(old,new)
    notebook = nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(script)],
                 metadata={"kernelspec": {"name": "ahr-go", "display_name": "AHR Go (.venv)", "language": "python"}})
    NotebookClient(notebook, timeout=180, kernel_name="ahr-go", resources={"metadata": {"path": str(project)}}).execute()
    return json.loads((Path(project) / "reports" / ("07_recovery_" + phase + ".json")).read_text())


In [ ]:
SOURCE_IDENTITIES = transfer_source_identity(PROJECT_ROOT)
SOURCE_DIGEST = transfer_digest(SOURCE_IDENTITIES)
RECOVERY_PROTOCOL = transfer_protocol(sources=SOURCE_IDENTITIES, models={}, candidates=("recovery_fixture",), seeds=(19,),
    budgets=(1.0,), max_moves=6, opponents=("recovery_fixture",), positions=False, label="recovery_fixture")
RECOVERY_RUN = "transfer-07-recovery-" + transfer_digest(RECOVERY_PROTOCOL)[:12]
interrupted = transfer_kernel_recovery(PROJECT_ROOT, RECOVERY_PROTOCOL, RECOVERY_RUN, "interrupt")
resumed = transfer_kernel_recovery(PROJECT_ROOT, RECOVERY_PROTOCOL, RECOVERY_RUN, "resume")
transfer_seed_recovery()
reference, reference_receipt = transfer_run(PROJECT_ROOT, RECOVERY_RUN + "-reference", RECOVERY_PROTOCOL,
    {"recovery_fixture": transfer_recovery_player}, {})
assert resumed["results"] == transfer_without_timing(reference["results"])
assert resumed["metrics"] == reference["metrics"]
assert all(row["status"] == "COMPLETED_TWO_PASSES" for row in reference["results"])
assert {row["candidate_color"] for row in reference["results"]} == {-1, 1}
RECOVERY_CHECK = {"status": "PASS", "fresh_kernels": 2, "exact_resumed_decisions_and_metrics": True,
    "timing_excluded_from_exact_comparison": True, "python_numpy_torch_rng_probes_identical": True,
    "nonempty_board_and_superko_history_restored": True, "resumed_from_stage": resumed["receipt"]["resumed_from_step"],
    "final_stage": reference["global_step"], "run_id": RECOVERY_RUN}
print(RECOVERY_CHECK)

In [ ]:
import tempfile, shutil
FAULT_CHECKS = []
with tempfile.TemporaryDirectory(prefix="transfer07-faults-", dir=PROJECT_ROOT / "checkpoints") as temporary:
    root = Path(temporary)
    original = PROJECT_ROOT / "checkpoints" / RECOVERY_RUN
    def expect_rejection(name, operation):
        try:
            operation()
        except (RuntimeError, ValueError, TypeError):
            FAULT_CHECKS.append(name)
        else:
            raise AssertionError("Expected rejection: " + name)
    broken = root / "corrupt"
    shutil.copytree(original, broken)
    manifest = json.loads((broken / "manifest.json").read_text())
    (broken / manifest["latest"]).write_bytes(b"broken")
    expect_rejection("corrupt_checkpoint_rejected", lambda: transfer_load_unlocked(broken, RECOVERY_PROTOCOL, {}))
    malformed = root / "malformed"
    shutil.copytree(original, malformed)
    (malformed / "manifest.json").write_text("{broken")
    expect_rejection("malformed_manifest_rejected", lambda: transfer_load_unlocked(malformed, RECOVERY_PROTOCOL, {}))
    changed = copy.deepcopy(RECOVERY_PROTOCOL); changed["komi"] = 1.5
    expect_rejection("changed_protocol_rejected", lambda: transfer_commit_protocol(original, changed))

    def protocol_for(name, max_moves=2):
        return transfer_protocol(sources=SOURCE_IDENTITIES, models={}, candidates=(name,), seeds=(31,),
            budgets=(0.001 if name == "slow_fixture" else 1.0,), max_moves=max_moves,
            opponents=("pass_fixture",), positions=False, label="recovery_fixture")
    def slow_fixture(state, budget, seed):
        time.sleep(budget + .01)
        return {"action": state.pass_action, "status": "SLOW_FIXTURE"}
    def illegal_fixture(state, budget, seed):
        return {"action": -1, "status": "ILLEGAL_FIXTURE"}
    def placement_fixture(state, budget, seed):
        return {"action": next(action for action in legal_actions(state) if action != state.pass_action), "status": "PLACEMENT_FIXTURE"}
    expected_status = {"slow_fixture": "UNKNOWN_TIMEOUT", "illegal_fixture": "ERROR_ILLEGAL_PROPOSAL", "placement_fixture": "UNKNOWN_MOVE_LIMIT"}
    for name, player in (("slow_fixture", slow_fixture), ("illegal_fixture", illegal_fixture), ("placement_fixture", placement_fixture)):
        protocol = protocol_for(name)
        # Disposable runs still live under this project's checkpoints directory.
        outcome, _ = transfer_run(PROJECT_ROOT, "transfer-07-fixture-" + transfer_digest(protocol)[:12], protocol,
            {name: player, "pass_fixture": transfer_baseline("pass_fixture")}, {})
        assert {row["status"] for row in outcome["results"]} == {expected_status[name]}
        assert all(row["candidate_score"] is None and row["winner"] is None for row in outcome["results"])
        FAULT_CHECKS.append(expected_status[name])

# Confirm independent rule/score checks on legal histories, including passing.
for seed in (13, 37):
    fixture = initial_state(3, 0.5)
    rng = random.Random(seed)
    for _ in range(20):
        assert area_score(fixture) == transfer_reference_score(fixture)
        if fixture.terminal:
            break
        for action in range(fixture.pass_action + 1):
            reference_board = transfer_reference_board(fixture, action)
            assert (reference_board is not None) == (action in legal_actions(fixture))
            if reference_board is not None:
                assert reference_board == play(fixture, action).board
        fixture = play(fixture, rng.choice(legal_actions(fixture)))
FAULT_CHECKS.append("independent_rule_and_score_agreement")
self_control_summary = transfer_summarize(RECOVERY_PROTOCOL, reference["results"])
assert self_control_summary["candidates"]["recovery_fixture"]["candidate_moves"] * 2 == self_control_summary["candidates"]["recovery_fixture"]["moves"]
FAULT_CHECKS.append("same_name_opponent_role_metrics_separated")
expect_rejection("duplicate_budgets_rejected", lambda: transfer_protocol(sources=SOURCE_IDENTITIES, models={}, candidates=("x",), budgets=(.1,.1)))
expect_rejection("duplicate_opponents_rejected", lambda: transfer_protocol(sources=SOURCE_IDENTITIES, models={}, candidates=("x",), opponents=("x","x")))
print({"status": "PASS", "checks": FAULT_CHECKS})

## Execute or resume frozen development evaluation

The default wiring runs use a newly initialized AHR and two swapped-color games on each of 3×3 and 9×9, capped at four moves. Any unfinished game remains UNKNOWN. With a separately produced CSP-only checkpoint, opt-in evaluation restores that checkpoint **and its original initialization**, fixes the protocol before playing, and compares both against the same controls. A 600-second session cap returns a resumable cursor; it does not turn an unfinished game into a loss. Rerun until `complete` is true.

Pilot overrides: `AHR07_SEEDS=1021,2053,3079`, `AHR07_BUDGETS=0.05,0.25`, `AHR07_MOVE_CAP=40` and `AHR07_BOARD_SIZE=3`. A 9×9 pilot uses project komi 7.5; small boards use 0.5. These values are committed into the protocol, so changing one creates a new experiment identity.

Terminal execution and exact resume use the same command and selected training run:

```text
AHR07_RUN_TRANSFER=1 AHR07_CSP_RUN=<completed-05-run-id> .venv/bin/python -m jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.kernel_name=ahr-go --ExecutePreprocessor.timeout=900 notebooks/07_csp_to_go_transfer_tests.ipynb
```

Preserve any custom override values when resuming. The protocol and model hashes must match; the evaluator never resets an invalid checkpoint.

In [ ]:
torch.manual_seed(20260923)
SMOKE_CODEC = ReasoningCodecConfig()
SMOKE_MODEL = NativeAHR(AHRConfig(max_tokens=SMOKE_CODEC.max_tokens)).cpu().eval()
SMOKE_SEARCH = AHRGoSearchConfig(max_depth=1, branch_width=2, max_nodes=8, queen_cycles=1)
SMOKE_MODELS = {"untrained_ahr": SMOKE_MODEL}
SMOKE_PLAYERS = {"untrained_ahr": transfer_ahr_player(SMOKE_MODEL, SMOKE_CODEC, SMOKE_SEARCH),
                 "random_legal": transfer_baseline("random_legal")}
SMOKE_PROTOCOL = transfer_protocol(sources=SOURCE_IDENTITIES,
    models={name: architecture_parameter_digest(model) for name,model in SMOKE_MODELS.items()},
    candidates=("untrained_ahr",), seeds=(941,), budgets=(1.0,), size=3, max_moves=4,
    opponents=("random_legal",), positions=True, label="wiring_smoke")
SMOKE_PROTOCOL["search_config"] = asdict(SMOKE_SEARCH)
SMOKE_PROTOCOL["codec_config"] = asdict(SMOKE_CODEC)
SMOKE_RUN = "transfer-07-wiring-" + transfer_digest(SMOKE_PROTOCOL)[:12]
smoke_state, smoke_receipt = transfer_run(PROJECT_ROOT, SMOKE_RUN, SMOKE_PROTOCOL, SMOKE_PLAYERS, SMOKE_MODELS)
assert smoke_state["complete"]
assert all(not row["status"].startswith("ERROR") for row in smoke_state["results"])
SMOKE_SUMMARY = transfer_summarize(SMOKE_PROTOCOL, smoke_state["results"])
print({"status": "PASS", "scope": "untrained development wiring only", "receipt": smoke_receipt,
       "outcomes": SMOKE_SUMMARY["status_counts"]})

SMOKE9_PROTOCOL = transfer_protocol(sources=SOURCE_IDENTITIES,
    models={name: architecture_parameter_digest(model) for name,model in SMOKE_MODELS.items()},
    candidates=("untrained_ahr",), seeds=(947,), budgets=(1.0,), size=9, komi=7.5, max_moves=4,
    opponents=("random_legal",), positions=False, label="wiring_smoke")
SMOKE9_PROTOCOL["search_config"] = asdict(SMOKE_SEARCH)
SMOKE9_PROTOCOL["codec_config"] = asdict(SMOKE_CODEC)
SMOKE9_RUN = "transfer-07-wiring9-" + transfer_digest(SMOKE9_PROTOCOL)[:12]
smoke9_state, smoke9_receipt = transfer_run(PROJECT_ROOT, SMOKE9_RUN, SMOKE9_PROTOCOL, SMOKE_PLAYERS, SMOKE_MODELS)
assert smoke9_state["complete"]
assert all(not row["status"].startswith("ERROR") for row in smoke9_state["results"])
SMOKE9_SUMMARY = transfer_summarize(SMOKE9_PROTOCOL, smoke9_state["results"])
print({"status": "PASS", "scope": "untrained 9x9 wiring; no strength claim", "outcomes": SMOKE9_SUMMARY["status_counts"]})

TRANSFER_RESULT = {"status": "NOT_RUN_NO_PRETRAINED_CHECKPOINT" if not SELECTED_CSP_RUN else "NOT_RUN_OPT_IN_REQUIRED",
                   "learned_transfer_established": False, "training_performed": False}
if TRANSFER_RUN_REQUESTED:
    if not SELECTED_CSP_RUN:
        raise RuntimeError("Set AHR07_CSP_RUN to a valid notebook05 CSP-only training run before requesting transfer")
    trained, original, codec, provenance = transfer_load_frozen_pair(PROJECT_ROOT, SELECTED_CSP_RUN)
    models = {"csp_pretrained_ahr": trained, "untrained_ahr": original}
    players = {name: transfer_ahr_player(model, codec) for name,model in models.items()}
    players.update({name: transfer_baseline(name) for name in ("random_legal", "greedy_area")})
    protocol = transfer_protocol(sources=SOURCE_IDENTITIES,
        models={name: architecture_parameter_digest(model) for name,model in models.items()},
        candidates=("csp_pretrained_ahr", "untrained_ahr", "random_legal", "greedy_area"),
        seeds=tuple(int(value) for value in os.environ.get("AHR07_SEEDS", "1021,2053,3079").split(",")),
        budgets=tuple(float(value) for value in os.environ.get("AHR07_BUDGETS", "0.05,0.25").split(",")),
        size=int(os.environ.get("AHR07_BOARD_SIZE", "3")),
        komi=7.5 if int(os.environ.get("AHR07_BOARD_SIZE", "3")) == 9 else 0.5,
        max_moves=int(os.environ.get("AHR07_MOVE_CAP", "40")),
        opponents=("random_legal", "greedy_area"), positions=True)
    # Checkpoint selection/codec/training dataset/source identities are precommitted too.
    protocol["pretraining_provenance"] = provenance
    protocol["search_config"] = asdict(AHRGoSearchConfig())
    protocol["codec_config"] = asdict(codec)
    run_id = "transfer-07-pilot-" + transfer_digest(protocol)[:12]
    evaluated, receipt = transfer_run(PROJECT_ROOT, run_id, protocol, players, models, session_seconds=600)
    TRANSFER_RESULT = {"status": "COMPLETED_DEVELOPMENT_PILOT" if evaluated["complete"] else "PAUSED_RESUMABLE_DEVELOPMENT_PILOT",
                       "learned_transfer_established": False, "training_performed": False, "receipt": receipt,
                       "summary": transfer_summarize(protocol, evaluated["results"]), "run_id": run_id}
    atomic_json(PROJECT_ROOT / "reports" / (run_id + ".json"), TRANSFER_RESULT)
print(TRANSFER_RESULT)

In [ ]:
VALIDATION_REPORT = {
    "schema": TRANSFER_SCHEMA, "status": "PASS", "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "sources": SOURCE_IDENTITIES, "training_performed": False, "optimizer_steps": 0,
    "lockbox_or_final_data_opened": False, "development_data": "new deterministic fixtures only",
    "recovery": RECOVERY_CHECK, "fault_and_semantic_checks": FAULT_CHECKS,
    "untrained_wiring": {"receipt": smoke_receipt, "summary": SMOKE_SUMMARY},
    "untrained_9x9_wiring": {"receipt": smoke9_receipt, "summary": SMOKE9_SUMMARY},
    "transfer_experiment": TRANSFER_RESULT,
    "limitations": ["No learned transfer result exists without a real CSP-only pretrained checkpoint.",
                    "Development smoke and pilot results cannot establish full-game optimality or a final transfer claim.",
                    "Wall-clock deadlines can change decisions after restart despite exact saved states.",
                    "UNKNOWN games do not become losses; complete-case estimates can be selection-biased.",
                    "3x3 development tests do not establish 9x9 playing strength."]}
atomic_json(PROJECT_ROOT / "reports/07_transfer_validation.json", VALIDATION_REPORT)
print({"validation": "PASS", "report": str(PROJECT_ROOT / "reports/07_transfer_validation.json"),
       "transfer_status": TRANSFER_RESULT["status"], "training": False})

## Interpretation and next authorized step

A passing receipt establishes that this evaluator is connected, frozen, auditable and resumable. It does **not** establish that CSP learning improves Go. That question requires an actual notebook05 CSP-only pretrained checkpoint, completed matched evaluations and a separately protected final test. The production board remains 9×9; the default small-board development fixtures keep validation bounded. No Go adaptation, self-play training, server or external publication starts here.